# 2402-01 · One bad query, three symptoms · solution

The reference version of the notebook you build with the lab guide, cell by cell. Open it if you get stuck.

**The question:** the top 100 customers by 2024 revenue, per tier and product category.

| Run | What changes | What it fixes |
|---|---|---|
| **A** | the bad query: function on the date column, unordered layout | nothing: 0 files pruned, heavy shuffle, skew |
| **B** | instruction 1 from Genie Code: filter on the raw column | still 0 pruned: the layout does not help yet |
| **C** | everything right: clustered layout, raw filter on both tables, aggregate before the skewed join | **read less, move less, balance more**: same answer as A, in a fraction of the time |

In [ ]:
%run "./2402-00-create-environment"

## 1 · Measure first: run A and open its Query Profile

In [ ]:
QUERY_A = """
-- A · the bad query, as someone wrote it: top 100 customers by 2024 revenue, per category
SELECT c.customer_id, c.tier, p.category,
       SUM(oi.quantity * p.list_price) AS revenue,
       COUNT(*)                        AS line_items
FROM lab2402.shop.orders o
JOIN lab2402.shop.order_items oi ON o.order_idx = oi.order_idx
JOIN lab2402.shop.products p     ON oi.product_idx = p.product_idx
JOIN lab2402.shop.customers c    ON o.customer_idx = c.customer_idx
WHERE date_format(o.order_date, 'yyyy-MM') BETWEEN '2024-01' AND '2024-12'
GROUP BY c.customer_id, c.tier, p.category
ORDER BY revenue DESC
LIMIT 100
"""

display(spark.sql(QUERY_A))

## 2 · Benchmark A

In [ ]:
benchmark("A", QUERY_A)

## 3 · Instruction 1: fix the predicate

In [ ]:
QUERY_B = """
-- B · instruction 1: the filter on the raw column, nothing else changes
SELECT c.customer_id, c.tier, p.category,
       SUM(oi.quantity * p.list_price) AS revenue,
       COUNT(*)                        AS line_items
FROM lab2402.shop.orders o
JOIN lab2402.shop.order_items oi ON o.order_idx = oi.order_idx
JOIN lab2402.shop.products p     ON oi.product_idx = p.product_idx
JOIN lab2402.shop.customers c    ON o.customer_idx = c.customer_idx
WHERE o.order_date >= DATE'2024-01-01' AND o.order_date < DATE'2025-01-01'
GROUP BY c.customer_id, c.tier, p.category
ORDER BY revenue DESC
LIMIT 100
"""

display(spark.sql(QUERY_B))

In [ ]:
benchmark("B", QUERY_B)

## 4 · Everything right: layout, filter and aggregation

In [ ]:
for t in ["orders", "order_items"]:
    spark.sql(f"ALTER TABLE lab2402.shop.{t} SET TBLPROPERTIES ('delta.targetFileSize' = '{LAYOUT_FILE_SIZE}')")
    spark.sql(f"ALTER TABLE lab2402.shop.{t} CLUSTER BY (order_date)")
    spark.sql(f"OPTIMIZE lab2402.shop.{t} FULL")      # recluster the existing data, not only new files
    file_stats(t)

In [ ]:
QUERY_C = """
-- C · everything right: raw filter on both tables, clustered layout,
--     and aggregate BEFORE the join with customers (the skewed one). Same answer as A.
WITH agg AS (
  SELECT o.customer_idx, p.category,
         SUM(oi.quantity * p.list_price) AS revenue,
         COUNT(*)                        AS line_items
  FROM lab2402.shop.orders o
  JOIN lab2402.shop.order_items oi ON o.order_idx = oi.order_idx
  JOIN lab2402.shop.products p     ON oi.product_idx = p.product_idx
  WHERE o.order_date  >= DATE'2024-01-01' AND o.order_date  < DATE'2025-01-01'
    AND oi.order_date >= DATE'2024-01-01' AND oi.order_date < DATE'2025-01-01'
  GROUP BY o.customer_idx, p.category
)
SELECT c.customer_id, c.tier, a.category, a.revenue, a.line_items
FROM agg a
JOIN lab2402.shop.customers c ON a.customer_idx = c.customer_idx
ORDER BY revenue DESC
LIMIT 100
"""

display(spark.sql(QUERY_C))

In [ ]:
benchmark("C", QUERY_C)

## 5 · The proof

In [ ]:
compare(QUERY_A, QUERY_C)